# VBS ASR trên Kaggle — tự động 1/2 GPU + báo cáo FPS

Notebook này chuyển pipeline Colab sang Kaggle và giữ định dạng transcript/keyframe của bản gốc.

- Đọc video trực tiếp hoặc từ ZIP trong `/kaggle/input`.
- Tự dùng tối đa 2 GPU; nếu không có GPU sẽ fallback CPU.
- Chia video theo tổng thời lượng để hai GPU có tải gần cân bằng.
- Dùng `BatchedInferencePipeline` để tăng throughput trên từng GPU.
- Đọc FPS thật của từng video bằng PyAV, không còn giả định tất cả video là 30 FPS.
- Xuất transcript, báo cáo FPS/tốc độ ASR, kết quả ghép keyframe (nếu có JSONL), và một file ZIP tổng hợp.

**Kaggle:** chọn **Settings → Accelerator → GPU**. Nếu model chưa nằm trong một Kaggle Dataset, bật Internet cho lần tải model đầu tiên. Thêm dataset chứa video/ZIP và JSONL bằng nút **Add Input**.


In [1]:
# Cài dependency. Faster-whisper dùng PyAV nên không cần apt-get/FFmpeg để giải mã audio.
%pip install -q -U "faster-whisper>=1.1.0" pandas


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 50.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.8/35.8 MB 36.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.5/39.5 MB 35.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 42.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
ydata-profiling 4.18.4 requires pandas!=1.4.0,<3.0,>1.5, but you have pandas 3.0.5 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2

## 1. Cấu hình

Giá trị mặc định cân bằng tốc độ/chất lượng cho `large-v3`. Nếu gặp thiếu VRAM, giảm `BATCH_SIZE` xuống 4 hoặc đổi `GPU_COMPUTE_TYPE` thành `int8_float16`.
`WORD_TIMESTAMPS=False` nhanh hơn và đủ cho output segment-level của notebook gốc.


In [2]:
import json
import math
import os
import shutil
import time
import zipfile
from bisect import bisect_right
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import av
import pandas as pd
import torch
from IPython.display import display

# ---------- Đường dẫn Kaggle ----------
INPUT_ROOT = Path("/kaggle/input/datasets/nguynminhphucaiwork/keyframe-l30")
OUTPUT_DIR = Path("/kaggle/working/vbs_asr_output")
EXTRACT_DIR = Path("/kaggle/working/vbs_asr_extracted")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Nếu video nằm ở một thư mục cụ thể, có thể thay INPUT_ROOT ở trên.
# Nếu có nhiều file JSONL, đặt đường dẫn cụ thể, ví dụ:
# KEYFRAMES_JSONL = Path("/kaggle/input/my-dataset/keyframes.jsonl")
KEYFRAMES_JSONL = Path("/kaggle/input/datasets/nguynminhphucaiwork/keyframe-l30/l30.jsonl")

# ---------- Model / hiệu năng ----------
# Có thể thay bằng đường dẫn model CTranslate2 trong /kaggle/input để chạy offline.
MODEL_NAME_OR_PATH = "large-v3"
MAX_GPUS = 2
USE_BATCHED = True
BATCH_SIZE = 8
GPU_COMPUTE_TYPE = "float16"
CPU_COMPUTE_TYPE = "int8"
CPU_THREADS = max(1, (os.cpu_count() or 4) - 1)

# ---------- Chất lượng ASR ----------
LANGUAGE = "vi"
BEAM_SIZE = 5
WORD_TIMESTAMPS = False
DEFAULT_FPS_IF_MISSING = 30.0
VIDEO_EXTENSIONS = {".mp4", ".mov", ".mkv", ".avi", ".webm", ".m4v", ".ts"}

GLOSSARY = (
    "Bản tin thời sự tiếng Việt. "
    "Các tên riêng, địa danh và tổ chức có thể gặp: "
    "Việt Nam, Hà Nội, Thành phố Hồ Chí Minh, Chính phủ, Quốc hội, VTV."
)

print("Cấu hình đã sẵn sàng.")


Cấu hình đã sẵn sàng.


## 2. Tìm video và giải nén ZIP (nếu cần)

Notebook ưu tiên video đã được Kaggle giải nén sẵn. Chỉ khi không tìm thấy video trực tiếp, nó mới giải nén các ZIP trong input. Tên file (không gồm phần mở rộng) tiếp tục được dùng làm `video_id`.


In [3]:
def safe_extract_zip(zip_path: Path, destination: Path) -> None:
    """Giải nén và chặn đường dẫn thoát ra ngoài destination."""
    destination.mkdir(parents=True, exist_ok=True)
    destination_resolved = destination.resolve()
    with zipfile.ZipFile(zip_path) as zf:
        for member in zf.infolist():
            target = (destination / member.filename).resolve()
            if target != destination_resolved and destination_resolved not in target.parents:
                raise ValueError(f"ZIP có đường dẫn không an toàn: {member.filename}")
        zf.extractall(destination)


def find_videos(root: Path):
    return sorted(
        p for p in root.rglob("*")
        if p.is_file() and p.suffix.lower() in VIDEO_EXTENSIONS
    )


VIDEO_PATHS = find_videos(INPUT_ROOT)

if not VIDEO_PATHS:
    zip_paths = sorted(INPUT_ROOT.rglob("*.zip"))
    if not zip_paths:
        raise FileNotFoundError(
            "Không thấy video hoặc ZIP trong /kaggle/input. Hãy Add Input rồi chạy lại."
        )

    if EXTRACT_DIR.exists():
        shutil.rmtree(EXTRACT_DIR)
    EXTRACT_DIR.mkdir(parents=True)

    for zip_path in zip_paths:
        target = EXTRACT_DIR / zip_path.stem
        print(f"Giải nén: {zip_path} -> {target}")
        safe_extract_zip(zip_path, target)

    VIDEO_PATHS = find_videos(EXTRACT_DIR)

if not VIDEO_PATHS:
    raise ValueError("Không tìm thấy video hợp lệ sau khi quét/giải nén.")

# Bản gốc dùng stem làm video_id; không cho phép ghi đè âm thầm khi trùng tên.
id_to_paths = {}
for path in VIDEO_PATHS:
    id_to_paths.setdefault(path.stem, []).append(path)
duplicate_ids = {k: v for k, v in id_to_paths.items() if len(v) > 1}
if duplicate_ids:
    details = "\n".join(f"- {k}: {[str(p) for p in v]}" for k, v in duplicate_ids.items())
    raise ValueError(f"Có video_id trùng nhau. Hãy đổi tên file trước khi chạy:\n{details}")

print(f"Tìm thấy {len(VIDEO_PATHS)} video.")
for path in VIDEO_PATHS[:20]:
    print("-", path)
if len(VIDEO_PATHS) > 20:
    print(f"... và {len(VIDEO_PATHS) - 20} video khác")


Tìm thấy 96 video.
- /kaggle/input/datasets/nguynminhphucaiwork/keyframe-l30/L30_V001.mp4
- /kaggle/input/datasets/nguynminhphucaiwork/keyframe-l30/L30_V002.mp4
- /kaggle/input/datasets/nguynminhphucaiwork/keyframe-l30/L30_V003.mp4
- /kaggle/input/datasets/nguynminhphucaiwork/keyframe-l30/L30_V004.mp4
- /kaggle/input/datasets/nguynminhphucaiwork/keyframe-l30/L30_V005.mp4
- /kaggle/input/datasets/nguynminhphucaiwork/keyframe-l30/L30_V006.mp4
- /kaggle/input/datasets/nguynminhphucaiwork/keyframe-l30/L30_V007.mp4
- /kaggle/input/datasets/nguynminhphucaiwork/keyframe-l30/L30_V008.mp4
- /kaggle/input/datasets/nguynminhphucaiwork/keyframe-l30/L30_V009.mp4
- /kaggle/input/datasets/nguynminhphucaiwork/keyframe-l30/L30_V010.mp4
- /kaggle/input/datasets/nguynminhphucaiwork/keyframe-l30/L30_V011.mp4
- /kaggle/input/datasets/nguynminhphucaiwork/keyframe-l30/L30_V012.mp4
- /kaggle/input/datasets/nguynminhphucaiwork/keyframe-l30/L30_V013.mp4
- /kaggle/input/datasets/nguynminhphucaiwork/keyframe-l30/

## 3. Đọc FPS và metadata video

`source_fps` là FPS gốc lấy từ stream video. Báo cáo cuối còn có `processing_fps` (số frame nguồn xử lý mỗi giây), `realtime_factor`, và `speed_x_realtime`.


In [4]:
def probe_video(path: Path):
    with av.open(str(path)) as container:
        stream = next((s for s in container.streams if s.type == "video"), None)
        if stream is None:
            raise ValueError("File không có video stream")

        rate = (
            stream.average_rate
            or getattr(stream, "base_rate", None)
            or getattr(stream, "guessed_rate", None)
        )
        fps = float(rate) if rate else None

        if stream.duration is not None and stream.time_base is not None:
            duration = float(stream.duration * stream.time_base)
        elif container.duration is not None:
            duration = float(container.duration / av.time_base)
        else:
            duration = None

        total_frames = int(stream.frames) if stream.frames else None
        if not total_frames and duration and fps:
            total_frames = int(round(duration * fps))

        return {
            "video_id": path.stem,
            "file_name": path.name,
            "path": str(path),
            "source_fps": round(fps, 6) if fps else None,
            "duration_sec": round(duration, 3) if duration is not None else None,
            "total_frames": total_frames,
            "width": stream.width,
            "height": stream.height,
            "codec": stream.codec_context.name,
        }


video_metadata = []
for video_path in VIDEO_PATHS:
    try:
        video_metadata.append(probe_video(video_path))
    except Exception as exc:
        video_metadata.append({
            "video_id": video_path.stem,
            "file_name": video_path.name,
            "path": str(video_path),
            "source_fps": None,
            "duration_sec": None,
            "total_frames": None,
            "width": None,
            "height": None,
            "codec": None,
            "probe_error": repr(exc),
        })

metadata_by_id = {row["video_id"]: row for row in video_metadata}
display(pd.DataFrame(video_metadata).drop(columns=["path"], errors="ignore"))


,video_id,file_name,source_fps,duration_sec,total_frames,width,height,codec
0,L30_V001,L30_V001.mp4,25.0,126.48,3162,1280,720,h264
1,L30_V002,L30_V002.mp4,25.0,164.68,4117,1280,720,h264
2,L30_V003,L30_V003.mp4,25.0,323.68,8092,1280,720,h264
3,L30_V004,L30_V004.mp4,25.0,240.08,6002,1280,720,h264
4,L30_V005,L30_V005.mp4,25.0,66.36,1659,1280,720,h264
...,...,...,...,...,...,...,...,...
91,L30_V092,L30_V092.mp4,25.0,173.40,4335,1280,720,h264
92,L30_V093,L30_V093.mp4,25.0,261.84,6546,1280,720,h264
93,L30_V094,L30_V094.mp4,25.0,219.08,5477,1280,720,h264
94,L30_V095,L30_V095.mp4,25.0,309.56,7739,1280,720,h264


## 4. Phát hiện GPU và tải model một lần vào cache

Với nhiều video, mỗi GPU nhận một nhóm file và chạy đồng thời. Nếu chỉ có đúng một video thì chỉ một GPU làm việc, vì `faster-whisper` không chia tự động một file đơn lẻ qua nhiều GPU.


In [5]:
cuda_count = torch.cuda.device_count()
available_gpu_ids = list(range(min(MAX_GPUS, cuda_count)))

if available_gpu_ids:
    for gpu_id in available_gpu_ids:
        props = torch.cuda.get_device_properties(gpu_id)
        print(f"GPU {gpu_id}: {props.name} ({props.total_memory / 2**30:.1f} GiB)")
else:
    print("Không phát hiện CUDA GPU — sẽ chạy CPU int8.")

model_candidate = Path(MODEL_NAME_OR_PATH)
if model_candidate.exists():
    RESOLVED_MODEL_PATH = str(model_candidate)
    print("Dùng model local:", RESOLVED_MODEL_PATH)
else:
    # Tải trước một lần để các worker GPU cùng đọc cache, tránh tải trùng/race.
    from faster_whisper.utils import download_model
    print(f"Đang chuẩn bị model {MODEL_NAME_OR_PATH!r}...")
    RESOLVED_MODEL_PATH = download_model(MODEL_NAME_OR_PATH)
    print("Model cache:", RESOLVED_MODEL_PATH)


GPU 0: Tesla T4 (14.6 GiB)
GPU 1: Tesla T4 (14.6 GiB)
Đang chuẩn bị model 'large-v3'...


Model cache: /root/.cache/huggingface/hub/models--Systran--faster-whisper-large-v3/snapshots/edaa852ec7e145841d8ffdb056a99866b5f0a478


## 5. Chạy ASR song song trên 1/2 GPU

Mỗi worker sở hữu model riêng trên đúng GPU của nó. Video dài được phân trước vào worker đang có tổng thời lượng thấp nhất để hạn chế một GPU xong quá sớm.


In [6]:
from faster_whisper import BatchedInferencePipeline, WhisperModel


def tc(seconds):
    ms = round(seconds * 1000)
    h, ms = divmod(ms, 3_600_000)
    m, ms = divmod(ms, 60_000)
    s, ms = divmod(ms, 1_000)
    return f"{h:02}:{m:02}:{s:02}.{ms:03}"


def make_shards(items, count):
    """Longest-processing-time greedy balancing theo duration."""
    shards = [[] for _ in range(count)]
    loads = [0.0] * count
    ordered = sorted(
        items,
        key=lambda row: row.get("duration_sec") or 0.0,
        reverse=True,
    )
    for item in ordered:
        index = min(range(count), key=loads.__getitem__)
        shards[index].append(item)
        loads[index] += item.get("duration_sec") or 0.0
    return shards, loads


def transcribe_shard(worker_index, items, gpu_id=None):
    device = "cuda" if gpu_id is not None else "cpu"
    compute_type = GPU_COMPUTE_TYPE if device == "cuda" else CPU_COMPUTE_TYPE
    label = f"GPU {gpu_id}" if gpu_id is not None else "CPU"

    load_started = time.perf_counter()
    model = WhisperModel(
        RESOLVED_MODEL_PATH,
        device=device,
        device_index=gpu_id or 0,
        compute_type=compute_type,
        cpu_threads=CPU_THREADS,
        num_workers=1,
    )
    engine = BatchedInferencePipeline(model=model) if USE_BATCHED else model
    model_load_sec = time.perf_counter() - load_started
    print(f"[{label}] Model sẵn sàng sau {model_load_sec:.1f}s; nhận {len(items)} video")

    transcripts = {}
    reports = []

    for item in items:
        video_id = item["video_id"]
        started = time.perf_counter()
        print(f"[{label}] Đang ASR: {video_id}")

        try:
            kwargs = dict(
                language=LANGUAGE,
                task="transcribe",
                beam_size=BEAM_SIZE,
                temperature=0.0,
                word_timestamps=WORD_TIMESTAMPS,
                without_timestamps=False,
                vad_filter=True,
                vad_parameters={
                    "threshold": 0.4,
                    "min_silence_duration_ms": 700,
                    "speech_pad_ms": 200,
                },
                repetition_penalty=1.15,
                no_repeat_ngram_size=3,
                initial_prompt=GLOSSARY,
            )
            if USE_BATCHED:
                kwargs["batch_size"] = BATCH_SIZE
            else:
                kwargs.update(
                    condition_on_previous_text=True,
                    hallucination_silence_threshold=(2.0 if WORD_TIMESTAMPS else None),
                )

            segments, info = engine.transcribe(item["path"], **kwargs)
            rows = []
            for seg in segments:  # inference thực sự chạy khi iterate generator
                text = seg.text.strip()
                if not text:
                    continue
                rows.append({
                    "id": len(rows),
                    "start": round(seg.start, 3),
                    "end": round(seg.end, 3),
                    "start_tc": tc(seg.start),
                    "end_tc": tc(seg.end),
                    "text_raw": text,
                    "text": text,
                })

            elapsed = time.perf_counter() - started
            duration = item.get("duration_sec") or getattr(info, "duration", None)
            total_frames = item.get("total_frames")
            transcripts[video_id] = rows
            reports.append({
                **item,
                "worker": label,
                "status": "ok",
                "segments": len(rows),
                "asr_elapsed_sec": round(elapsed, 3),
                "processing_fps": round(total_frames / elapsed, 3) if total_frames else None,
                "realtime_factor": round(elapsed / duration, 4) if duration else None,
                "speed_x_realtime": round(duration / elapsed, 3) if duration else None,
                "error": None,
            })
            print(
                f"[{label}] Xong {video_id}: {len(rows)} đoạn | "
                f"{elapsed:.1f}s | {duration / elapsed:.2f}x realtime"
                if duration else
                f"[{label}] Xong {video_id}: {len(rows)} đoạn | {elapsed:.1f}s"
            )
        except Exception as exc:
            elapsed = time.perf_counter() - started
            transcripts[video_id] = []
            reports.append({
                **item,
                "worker": label,
                "status": "error",
                "segments": 0,
                "asr_elapsed_sec": round(elapsed, 3),
                "processing_fps": None,
                "realtime_factor": None,
                "speed_x_realtime": None,
                "error": repr(exc),
            })
            print(f"[{label}] LỖI {video_id}: {exc!r}")

    return transcripts, reports, {"worker": label, "model_load_sec": round(model_load_sec, 3)}


worker_count = min(
    len(video_metadata),
    len(available_gpu_ids) if available_gpu_ids else 1,
)
shards, estimated_loads = make_shards(video_metadata, worker_count)
print("Phân tải ước tính (giây video):", [round(x, 1) for x in estimated_loads])

all_asr = {}
asr_reports = []
worker_reports = []
end_to_end_started = time.perf_counter()

if worker_count == 1:
    gpu_id = available_gpu_ids[0] if available_gpu_ids else None
    transcript_part, report_part, worker_part = transcribe_shard(0, shards[0], gpu_id)
    all_asr.update(transcript_part)
    asr_reports.extend(report_part)
    worker_reports.append(worker_part)
else:
    with ThreadPoolExecutor(max_workers=worker_count) as executor:
        futures = {
            executor.submit(transcribe_shard, i, shards[i], available_gpu_ids[i]): i
            for i in range(worker_count)
        }
        for future in as_completed(futures):
            transcript_part, report_part, worker_part = future.result()
            all_asr.update(transcript_part)
            asr_reports.extend(report_part)
            worker_reports.append(worker_part)

end_to_end_wall_sec = time.perf_counter() - end_to_end_started
all_asr = dict(sorted(all_asr.items()))
asr_reports.sort(key=lambda row: row["video_id"])

ok_reports = [row for row in asr_reports if row["status"] == "ok"]
total_frames_ok = sum(row.get("total_frames") or 0 for row in ok_reports)
total_duration_ok = sum(row.get("duration_sec") or 0 for row in ok_reports)
run_summary = {
    "model": MODEL_NAME_OR_PATH,
    "batched": USE_BATCHED,
    "batch_size": BATCH_SIZE if USE_BATCHED else None,
    "compute_type": GPU_COMPUTE_TYPE if available_gpu_ids else CPU_COMPUTE_TYPE,
    "visible_gpu_count": cuda_count,
    "workers_used": worker_count,
    "videos_total": len(asr_reports),
    "videos_ok": len(ok_reports),
    "videos_error": len(asr_reports) - len(ok_reports),
    "total_video_duration_sec": round(total_duration_ok, 3),
    "end_to_end_wall_sec_including_model_load": round(end_to_end_wall_sec, 3),
    "effective_processing_fps_including_model_load": (
        round(total_frames_ok / end_to_end_wall_sec, 3) if total_frames_ok else None
    ),
    "effective_speed_x_realtime_including_model_load": (
        round(total_duration_ok / end_to_end_wall_sec, 3) if total_duration_ok else None
    ),
    "worker_model_load": sorted(worker_reports, key=lambda row: row["worker"]),
}

print(json.dumps(run_summary, ensure_ascii=False, indent=2))
display(pd.DataFrame(asr_reports).drop(columns=["path"], errors="ignore"))


Phân tải ước tính (giây video): [10232.8, 10238.6]
[GPU 1] Model sẵn sàng sau 3.3s; nhận 48 video
[GPU 1] Đang ASR: L30_V048
[GPU 0] Model sẵn sàng sau 3.5s; nhận 48 video
[GPU 0] Đang ASR: L30_V009
[GPU 1] Xong L30_V048: 26 đoạn | 32.8s | 12.19x realtime
[GPU 1] Đang ASR: L30_V018
[GPU 0] Xong L30_V009: 53 đoạn | 37.2s | 11.32x realtime
[GPU 0] Đang ASR: L30_V054
[GPU 0] Xong L30_V054: 26 đoạn | 16.3s | 22.49x realtime
[GPU 0] Đang ASR: L30_V030
[GPU 1] Xong L30_V018: 34 đoạn | 21.2s | 17.84x realtime
[GPU 1] Đang ASR: L30_V027
[GPU 1] Xong L30_V027: 34 đoạn | 21.8s | 16.10x realtime
[GPU 1] Đang ASR: L30_V033
[GPU 0] Xong L30_V030: 49 đoạn | 23.0s | 14.94x realtime
[GPU 0] Đang ASR: L30_V061
[GPU 1] Xong L30_V033: 19 đoạn | 11.4s | 30.23x realtime
[GPU 1] Đang ASR: L30_V003
[GPU 0] Xong L30_V061: 28 đoạn | 18.8s | 17.71x realtime
[GPU 0] Đang ASR: L30_V045
[GPU 1] Xong L30_V003: 39 đoạn | 26.3s | 12.29x realtime
[GPU 1] Đang ASR: L30_V029
[GPU 1] Xong L30_V029: 0 đoạn | 1.8s | 170.57

,video_id,file_name,source_fps,duration_sec,total_frames,width,height,codec,worker,status,segments,asr_elapsed_sec,processing_fps,realtime_factor,speed_x_realtime,error
0,L30_V001,L30_V001.mp4,25.0,126.48,3162,1280,720,h264,GPU 1,ok,14,10.259,308.205,0.0811,12.328,None
1,L30_V002,L30_V002.mp4,25.0,164.68,4117,1280,720,h264,GPU 0,ok,16,13.174,312.508,0.0800,12.500,None
2,L30_V003,L30_V003.mp4,25.0,323.68,8092,1280,720,h264,GPU 1,ok,39,26.338,307.236,0.0814,12.289,None
3,L30_V004,L30_V004.mp4,25.0,240.08,6002,1280,720,h264,GPU 1,ok,22,15.546,386.083,0.0648,15.443,None
4,L30_V005,L30_V005.mp4,25.0,66.36,1659,1280,720,h264,GPU 0,ok,8,6.999,237.047,0.1055,9.482,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
91,L30_V092,L30_V092.mp4,25.0,173.40,4335,1280,720,h264,GPU 1,ok,14,11.838,366.182,0.0683,14.647,None
92,L30_V093,L30_V093.mp4,25.0,261.84,6546,1280,720,h264,GPU 1,ok,20,17.383,376.569,0.0664,15.063,None
93,L30_V094,L30_V094.mp4,25.0,219.08,5477,1280,720,h264,GPU 0,ok,24,15.537,352.512,0.0709,14.100,None
94,L30_V095,L30_V095.mp4,25.0,309.56,7739,1280,720,h264,GPU 0,ok,31,18.327,422.273,0.0592,16.891,None


## 6. Lưu transcript và báo cáo FPS

- `all_transcripts.json`: cùng cấu trúc chính như notebook cũ.
- `video_asr_report.csv/json`: FPS gốc và tốc độ xử lý theo từng video.
- `run_summary.json`: throughput tổng thể, số GPU/worker và cấu hình.


In [7]:
transcripts_path = OUTPUT_DIR / "all_transcripts.json"
report_json_path = OUTPUT_DIR / "video_asr_report.json"
report_csv_path = OUTPUT_DIR / "video_asr_report.csv"
summary_path = OUTPUT_DIR / "run_summary.json"

with transcripts_path.open("w", encoding="utf-8") as f:
    json.dump(all_asr, f, ensure_ascii=False, indent=2)

# Không lặp absolute path của Kaggle trong artifact để report dễ di chuyển.
portable_reports = [
    {k: v for k, v in row.items() if k != "path"}
    for row in asr_reports
]
with report_json_path.open("w", encoding="utf-8") as f:
    json.dump(portable_reports, f, ensure_ascii=False, indent=2)
pd.DataFrame(portable_reports).to_csv(report_csv_path, index=False, encoding="utf-8-sig")
with summary_path.open("w", encoding="utf-8") as f:
    json.dump(run_summary, f, ensure_ascii=False, indent=2)

print("Đã lưu:")
for path in (transcripts_path, report_json_path, report_csv_path, summary_path):
    print("-", path)


Đã lưu:
- /kaggle/working/vbs_asr_output/all_transcripts.json
- /kaggle/working/vbs_asr_output/video_asr_report.json
- /kaggle/working/vbs_asr_output/video_asr_report.csv
- /kaggle/working/vbs_asr_output/run_summary.json


## 7. Ghép transcript vào keyframe JSONL (tùy chọn)

Nếu tìm thấy đúng một JSONL, notebook tự ghép bằng `frame_idx / source_fps` của từng video. Nếu metadata không có FPS, nó mới fallback về 30 FPS. Việc tìm segment dùng binary search, nhanh hơn quét tuyến tính toàn bộ transcript cho mỗi keyframe.


In [8]:
def discover_jsonl():
    if KEYFRAMES_JSONL is not None:
        path = Path(KEYFRAMES_JSONL)
        if not path.exists():
            raise FileNotFoundError(f"Không thấy KEYFRAMES_JSONL: {path}")
        return path

    roots = [INPUT_ROOT]
    if EXTRACT_DIR.exists():
        roots.append(EXTRACT_DIR)
    candidates = sorted({p.resolve() for root in roots for p in root.rglob("*.jsonl")})
    if len(candidates) == 1:
        return candidates[0]
    if len(candidates) > 1:
        print("Có nhiều JSONL; hãy đặt KEYFRAMES_JSONL ở cell cấu hình:")
        for path in candidates:
            print("-", path)
    else:
        print("Không có JSONL; bỏ qua bước ghép keyframe.")
    return None


keyframes_jsonl = discover_jsonl()
keyframes_output_path = None
if keyframes_jsonl is not None:
    segment_index = {
        video_id: ([seg["start"] for seg in segments], segments)
        for video_id, segments in all_asr.items()
    }

    # Convert the ASR video_id to lowercase using .lower()
    output_rows = []
    skipped_no_video = 0
    skipped_no_text = 0
    fallback_fps_count = 0
    total_keyframes = 0

    with keyframes_jsonl.open("r", encoding="utf-8") as f:
        for line_number, line in enumerate(f, start=1):
            if not line.strip():
                continue
            total_keyframes += 1
            keyframe = json.loads(line)
            
            video_id = keyframe["video_id"].upper().replace('-','_')
            frame_idx =keyframe["frame_id"]
            shot_id = keyframe["shot_id"]
            if frame_idx is None:
                raise KeyError(f"Dòng {line_number} thiếu frame_idx/frames_id")

            indexed = segment_index.get(video_id)
            if indexed is None:
                skipped_no_video += 1
                continue

            fps = metadata_by_id.get(video_id, {}).get("source_fps")
            
            if not fps or not math.isfinite(fps) or fps <= 0:
                fps = DEFAULT_FPS_IF_MISSING
                fallback_fps_count += 1
            frame_time = float(frame_idx) / fps

            starts, segments = indexed
            position = bisect_right(starts, frame_time) - 1
            segment = segments[position] if position >= 0 else None
            video_id = keyframe["video_id"].lower().replace('_','-')
            if segment is None or frame_time > segment["end"]:
                skipped_no_text += 1
                
                output_rows.append({
                "video_id": video_id,
                "shot_id" : shot_id,
                "frames_id": frame_idx,
                "acr_text": "",
            })
            else :
                output_rows.append({
                "video_id": video_id,
                "shot_id": shot_id,
                "frames_id": frame_idx,
                "acr_text": segment["text"],
            })

    output_rows.sort(key=lambda x: (x["video_id"], x["frames_id"]))
    keyframes_output_path = OUTPUT_DIR / "keyframes_asr.json"
    with keyframes_output_path.open("w", encoding="utf-8") as f:
        json.dump(output_rows, f, ensure_ascii=False, indent=2)

    print(f"Tổng keyframe: {total_keyframes}")
    print(f"Đã ghép: {len(output_rows)}")
    print(f"Không có transcript video: {skipped_no_video}")
    print(f"Không nằm trong đoạn lời thoại: {skipped_no_text}")
    print(f"Keyframe phải fallback FPS={DEFAULT_FPS_IF_MISSING}: {fallback_fps_count}")
    print("Output:", keyframes_output_path)


Tổng keyframe: 19333
Đã ghép: 19333
Không có transcript video: 0
Không nằm trong đoạn lời thoại: 2638
Keyframe phải fallback FPS=30.0: 0
Output: /kaggle/working/vbs_asr_output/keyframes_asr.json


## 8. Đóng gói kết quả

File ZIP xuất hiện trong tab **Output** của Kaggle sau khi notebook chạy xong.


In [9]:
archive_base = Path("/kaggle/working/vbs_asr_results")
archive_path = Path(shutil.make_archive(str(archive_base), "zip", root_dir=OUTPUT_DIR))

print("Hoàn tất:", archive_path)
print("Các file trong gói:")
with zipfile.ZipFile(archive_path) as zf:
    for name in zf.namelist():
        print("-", name)

try:
    from IPython.display import FileLink, display
    display(FileLink(str(archive_path)))
except Exception:
    pass


Hoàn tất: /kaggle/working/vbs_asr_results.zip
Các file trong gói:
- run_summary.json
- video_asr_report.json
- all_transcripts.json
- video_asr_report.csv
- keyframes_asr.json


/kaggle/working/vbs_asr_results.zip